## Run auto syllable segmentation


In [ ]:
from pathlib import Path
import util as ut
import sys
import os

sys.path.append("/home/alily/Documents/GitHub/OnsetOffsetSpeech")
import inference_W
import pandas as pd
import save_in_TextGrid_silent_included

checkpoint = "/home/alily/Documents/GitHub/OnsetOffsetSpeech/checkpoints/best.pth"
data_dir = "/home/alily/Documents/GitHub/TongueTwister/Experiments/TT1/data"
sub_list = [
                'sub-01'
            ]


all_trials = [] 

for sub in sub_list:
        
        sub_dir = Path(data_dir)/sub
        info_file = sub_dir / f"behav_{sub}.tsv"
        runs_info = ut.get_runs_info(info_file)      # runs metadata
        wav_files = list(sub_dir.glob("*.wav"))         # wav files


        for file in wav_files:
                _, filename = os.path.split(file)
                # get trial meta data
                trial_info, syllables =ut.get_trial_info(file, runs_info)
                
                # run segmentation
                onset_times, offset_times = inference_W.main(
                        WAV_PATH=file,
                        CHECKPOINT=checkpoint,)
                save_in_TextGrid_silent_included.save_onset_offset_textgrid(file, onset_times, offset_times, output_path=f'{sub_dir}/{file.stem}.TextGrid')
                 
                # arrange results in df
                trial_df = ut.make_speech_trial_out(sub, trial_info, syllables, onset_times, offset_times)
                all_trials.append(trial_df)
                
results_df = pd.concat(all_trials, ignore_index=True)
#results_df.to_csv(r'Z:\data\Articulation\Sequence1\behavioral\auto\model_output_orig.tsv',sep='\t',index=False)
results_df.to_csv('/home/alily/Documents/GitHub/TongueTwister/Experiments/TT1/data/model_output_orig.tsv',sep='\t',index=False)

DEVICE: cpu


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 12064.64it/s]
/home/alily/Documents/GitHub/TongueTwister/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.35 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


FileNotFoundError: [Errno 2] No such file or directory: '/home/alily/Documents/GitHub/OnsetOffsetSpeech/checkpoints/best.pth'

In [20]:
import os
_, filename = os.path.split(file)
filename

'behav_sub-00_run-00_stim-44_pi pi pi pi pi pi pi pi pi .wav'

In [ ]:
# add columns to the data file
all_data = pd.read_csv(r'Z:\data\Articulation\Sequence1\behavioral\auto\model_output_orig.tsv',sep='\t')

all_data["isError"] = 0
all_data["isSyll"] = 1

# syllable time: onset to offset
all_data["ST"] = all_data["offset"]-all_data["onset"]

# inter-syllable interval: Onset-to-onset interval
all_data["ISI"] = (
    all_data.groupby(["SubNum", "BN", "TN"])["onset"].shift(-1)
    - all_data["onset"]
)

# gap: Offset-to-next-onset gap
all_data["gap"] = (
    all_data.groupby(["SubNum", "BN", "TN"])["onset"].shift(-1)
    - all_data["offset"]
)

# save
all_data.to_csv(r'Z:\data\Articulation\Sequence1\behavioral\auto\alldata.tsv',sep='\t',index=False)
